# Tests: `fasterai.core.parametrize` (source `nbs/core/parametrize.ipynb`)

In [ ]:
from fastcore.test import *
import copy, io
import torch
import torch.nn as nn
from torch.nn.utils import parametrize
from fasterai.core.parametrize import _is_parametrized, _master, _plain_modules, _unparametrize

In [ ]:
class _Double(nn.Module):
    "A parametrization with a visible effect: the weight is twice the master"
    def forward(self, w): return w * 2

def _conv():
    torch.manual_seed(0)
    return nn.Conv2d(3, 4, 3)

# a plain module: the master IS the parameter, and dropping a parametrization it has not got is a no-op
_c = _conv()
test_eq(_is_parametrized(_c), False)
test_is(_master(_c), _c.weight)
_unparametrize(_c)
test_is(_master(_c), _c.weight)
test_eq(sorted(_c.state_dict()), ['bias', 'weight'])

# once parametrized, the two come apart: the weight is computed, the master keeps its identity
_c = _conv()
_p0 = _c.weight
parametrize.register_parametrization(_c, 'weight', _Double())
test_eq(_is_parametrized(_c), True)
test_is(_master(_c), _p0)
test_eq(isinstance(_master(_c), nn.Parameter), True)
test_eq(torch.equal(_c.weight, _p0 * 2), True)
assert 'parametrizations.weight.original' in _c.state_dict(), list(_c.state_dict())
test_eq('weight' in _c.state_dict(), False)

# the trap these helpers exist for: a write to the computed weight is silently discarded
with torch.no_grad(): _c.weight.copy_(torch.zeros_like(_c.weight))
assert float(_c.weight.abs().sum()) > 0, 'the write landed: this trap is gone, and `_master` with it'
with torch.no_grad(): _master(_c).copy_(torch.zeros_like(_master(_c)))
test_eq(float(_c.weight.abs().sum()), 0.0)

In [ ]:
# leave_parametrized=True bakes the computed weight in, on the SAME parameter object
_c = _conv()
_id0 = id(_c.weight)
_expected = _c.weight.detach().clone() * 2
parametrize.register_parametrization(_c, 'weight', _Double())
_unparametrize(_c, leave_parametrized=True)
test_eq(_is_parametrized(_c), False)
test_eq(id(_c.weight), _id0)
test_eq(isinstance(_c.weight, nn.Parameter), True)
test_eq(torch.equal(_c.weight.detach(), _expected), True)
test_eq(sorted(_c.state_dict()), ['bias', 'weight'])

# leave_parametrized=False (the default) drops the rounding and restores the master
_c = _conv()
_master0 = _c.weight.detach().clone()
parametrize.register_parametrization(_c, 'weight', _Double())
_unparametrize(_c)
test_eq(torch.equal(_c.weight.detach(), _master0), True)

# an optimizer built BEFORE the parametrization still trains the master: same object, so no rebinding
_c = _conv()
_opt = torch.optim.SGD(_c.parameters(), lr=0.1)
_held = {id(p) for group in _opt.param_groups for p in group['params']}
parametrize.register_parametrization(_c, 'weight', _Double())
assert id(_master(_c)) in _held, 'the optimizer no longer holds the parameter being trained'
_before = _master(_c).detach().clone()
_c(torch.randn(1, 3, 8, 8)).sum().backward()
assert _master(_c).grad is not None, 'no gradient reached the master'
_opt.step()
test_ne(_master(_c).detach().tolist(), _before.tolist())

# a module whose weight is None reads back as None, not as an error
test_eq(_master(nn.BatchNorm2d(4, affine=False)), None)

In [ ]:
# --- a copy can be unparametrized without breaking the model it was copied from ---
# torch injects a per-module class to carry the `weight` property, and `copy.deepcopy` hands the SAME
# class to the copy. `remove_parametrizations` deletes the property from that class, which would leave
# the live model with no `weight` at all — this is why `_unparametrize` gives the plain class back
# instead of deleting anything.
_m = nn.Sequential(nn.Conv2d(3, 4, 3), nn.Flatten(), nn.Linear(4 * 6 * 6, 2))
parametrize.register_parametrization(_m[0], 'weight', _Double())
_x = torch.randn(1, 3, 8, 8)
_ref = _m(_x).detach().clone()
_master0 = _master(_m[0]).detach().clone()

_copy = copy.deepcopy(_m)
_unparametrize(_copy[0])
test_eq(_is_parametrized(_copy[0]), False)
test_eq(type(_copy[0]).__name__, 'Conv2d')
test_eq(torch.equal(_copy[0].weight.detach(), _master0), True)
torch.save(_copy, io.BytesIO())          # ...and a plain copy serializes, where a parametrized one raises
test_eq(_is_parametrized(_m[0]), True)   # the live model is untouched
test_eq(torch.equal(_m(_x), _ref), True)

# the same on the live model, and it goes on working afterwards
_unparametrize(_m[0], leave_parametrized=True)
test_eq(type(_m[0]).__name__, 'Conv2d')
test_eq(torch.equal(_m(_x), _ref), True)
torch.save(_m, io.BytesIO())
_m[0].weight.sum().backward()
assert _m[0].weight.grad is not None, 'the restored parameter no longer trains'

# a module that also parametrizes another tensor is left to torch, and keeps that parametrization
_c = _conv()
parametrize.register_parametrization(_c, 'weight', _Double())
parametrize.register_parametrization(_c, 'bias', _Double())
_unparametrize(_c)
test_eq(_is_parametrized(_c), False)
test_eq(parametrize.is_parametrized(_c, 'bias'), True)
test_eq(_c(_x).shape, (1, 4, 6, 6))

# `_plain_modules` skips the containers a parametrization inserts, so a positional walk still pairs a
# convolution with the module registered after it
_seq = nn.Sequential(nn.Conv2d(3, 4, 3), nn.BatchNorm2d(4), nn.ReLU())
_before = list(_seq.modules())
parametrize.register_parametrization(_seq[0], 'weight', _Double())
assert len(list(_seq.modules())) > len(_before), 'the parametrization inserted no module: check the fixture'
test_eq(list(_plain_modules(_seq)), _before)
_plain = list(_plain_modules(_seq))
test_eq(isinstance(_plain[_plain.index(_seq[0]) + 1], nn.BatchNorm2d), True)